# Semana 10 — Exercícios de Aula: SQL Avançado e Performance

Notebook organizado **na ordem de aparição dos slides da Semana 10**.

Cada bloco indica o número do slide de origem e traz:

- **Exemplos** adaptados para o modelo de e-commerce usado em aula.
- **Exercícios** para os alunos resolverem.
- Consultas usando `fato_vendas`, `dim_cliente`, `dim_produto` e `dim_tempo`.

## Tabelas consideradas

### `fato_vendas`
Tabela fato com as vendas.

Colunas usadas nos exercícios:
- `venda_sk`
- `order_id`
- `cliente_sk`
- `produto_sk`
- `tempo_sk`
- `valor_produto`
- `valor_frete`
- `valor_total`

### `dim_cliente`
Dimensão de clientes.

Colunas usadas:
- `cliente_sk`
- `customer_city`
- `customer_state`

### `dim_produto`
Dimensão de produtos.

Colunas usadas:
- `produto_sk`
- `product_category_name_english`

### `dim_tempo`
Dimensão de tempo.

Colunas usadas:
- `tempo_sk`
- `data`
- `ano`
- `mes`
- `trimestre`
- `ano_mes`

## Preparação

Use este padrão para executar SQL no notebook, caso esteja usando DuckDB.

Se as tabelas já estiverem carregadas no ambiente, siga direto para os exemplos.

In [22]:
import duckdb
import pandas as pd


In [23]:
dim_cliente = pd.read_csv('F:\Estudos\Senai_dados\Semana_10\Dataset_schema/dim_cliente.csv')
dim_produto = pd.read_csv('F:\Estudos\Senai_dados\Semana_10\Dataset_schema/dim_produto.csv')
dim_tempo = pd.read_csv('F:\Estudos\Senai_dados\Semana_10\Dataset_schema/dim_tempo.csv')
fato_vendas = pd.read_csv('F:\Estudos\Senai_dados\Semana_10\Dataset_schema/fato_vendas.csv')

<>:1: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:2: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:3: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:4: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:1: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:2: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:3: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not w

In [24]:
banco_dados = duckdb.connect()
banco_dados.register('dim_cliente', dim_cliente) # registra a dimensão cliente no banco de dados
banco_dados.register('dim_produto', dim_produto) # registra a dimensão produto no banco de dados
banco_dados.register('dim_tempo', dim_tempo) # registra a dimensão tempo no banco de dados
banco_dados.register('fato_vendas', fato_vendas) # registra a tabela fato vendas no banco de dados

---

# Slide 3 — Exemplo: Subconsultas

O slide apresenta subconsultas como consultas dentro de consultas.  
Aqui o exemplo foi adaptado para o nosso modelo de e-commerce.

## Exemplo — Vendas acima da média geral

Pergunta de negócio:

> Quais vendas tiveram valor total acima da média geral de vendas?

In [25]:
query = """
SELECT AVG(valor_total) AS media_geral
FROM fato_vendas
"""

banco_dados.sql(query).df()

,media_geral
0,139.929161


In [26]:
query = """
SELECT
    venda_sk,
    valor_total
FROM fato_vendas
WHERE valor_total > (
    SELECT AVG(valor_total)
    FROM fato_vendas
)
ORDER BY valor_total DESC;
"""

banco_dados.sql(query).df()

,venda_sk,valor_total
0,3482,6929.31
1,109785,6922.21
2,105496,6726.66
3,72721,4950.34
4,10998,4764.34
...,...,...
32858,62148,139.93
32859,62149,139.93
32860,93983,139.93
32861,81520,139.93


---

# Slide 4 — Exercícios: Subconsultas na prática

Resolva usando subconsultas.

## Slide 4 — Fácil: Vendas acima da média geral

Liste as vendas com `valor_total` acima da média geral.

Retorne:
- `venda_sk`
- `order_id`
- `valor_total`

Ordene por `valor_total` decrescente.

In [27]:
# Resposta do aluno

query = """
SELECT
    venda_sk,
    order_id,
    valor_total
FROM fato_vendas
WHERE valor_total > (SELECT AVG(valor_total)FROM fato_vendas)
ORDER BY valor_total

"""

banco_dados.sql(query).df()

,venda_sk,order_id,valor_total
0,106777,f814e911eb0f9834305ddacbaa7e83e8,139.93
1,10330,1823483895a111ad1b46e1c7e3cb7c81,139.93
2,60872,8e1c983bb7b5583c8b0e8a2d2f47677e,139.93
3,93983,d9f134a34fb4b3cfd95550b4785e8d05,139.93
4,62148,911c0c489bf4354f38fc64491075e8ba,139.93
...,...,...,...
32858,10998,199af31afc78c699f0dbf71fb178d4d4,4764.34
32859,72721,a96610ab360d42a2e5335a3998b4718a,4950.34
32860,105496,f5136e38d1a14a4dbd87dff67da82701,6726.66
32861,109785,fefacc66af859508bf1a7934eab1e97f,6922.21


## Slide 4 — Médio — Estados com volume acima da média

Calcule a quantidade de pedidos por estado e liste apenas os estados com quantidade de pedidos acima da média dos estados.

Use:
- `fato_vendas`
- `dim_cliente`

Retorne:
- `estado`
- `qtd_pedidos`

In [40]:
# Resposta do aluno

query = """
-- coloque sua resposta aqui
SELECT
    c.customer_state as estado,
    COUNT(f.order_id) as qtd_pedidos
FROM fato_vendas f
JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY c.customer_state
HAVING qtd_pedidos > (SELECT
                                AVG(qtd_pedidos)
                            FROM (SELECT
                                    c.customer_state as estado,
                                    COUNT(f.order_id) as qtd_pedidos
                                FROM fato_vendas f
                                JOIN dim_cliente c
                                    ON f.cliente_sk = c.cliente_sk
                                GROUP BY c.customer_state))
ORDER BY estado







"""



banco_dados.sql(query).df()

,estado,qtd_pedidos
0,MG,12916
1,PR,5649
2,RJ,14143
3,RS,6134
4,SC,4097
5,SP,46448


## Slide 4 — Difícil — Vendas das 3 categorias com maior receita

Retorne as vendas pertencentes às 3 categorias com maior receita total.

Use uma subconsulta para descobrir as 3 categorias.

Retorne:
- `venda_sk`
- `order_id`
- `categoria`
- `valor_total`

Dica:
Use esta subconsulta:  
WHERE v.produto_sk IN (

-- SUB-CONSULTA: Agrupa por produto e conta a quantidade de linhas (pedidos)  
-- Ordena do mais vendido para o menos vendido e limita aos 3 primeiros  

SELECT produto_sk  
FROM fato_vendas  
GROUP BY produto_sk  
ORDER BY COUNT(order_id) DESC   
LIMIT 3). 


In [ ]:
# Resposta do aluno
query = """
SELECT produto_sk  
FROM fato_vendas  
JOIN 
    ON
GROUP BY product_category_name  
ORDER BY COUNT(order_id) DESC   
LIMIT 3







"""

banco_dados.sql(query).df()


---

# Slide 6 — Exemplo: Common Table Expression, CTE

O slide apresenta CTE como uma consulta nomeada criada com `WITH`.

## Exemplo — Vendas de SP como CTE

Pergunta de negócio:

> Como criar uma etapa intermediária com as vendas de clientes de SP?

In [ ]:
query = """
WITH vendas_sp AS (
    SELECT
        f.venda_sk,
        f.order_id,
        f.valor_total,
        c.customer_state AS estado
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    WHERE c.customer_state = 'SP'
)
SELECT *
FROM vendas_sp
ORDER BY valor_total DESC
LIMIT 20;
"""

banco_dados.sql(query).df()

---

# Slide 7 — Exemplo: Sintaxe de CTE e múltiplas CTEs

O slide mostra a estrutura básica de uma CTE e o uso de múltiplas CTEs.

In [ ]:
# Exemplo do Slide 7 com múltiplas CTEs

query = """
WITH receita_por_categoria AS (
    SELECT
        p.product_category_name_english AS categoria,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    GROUP BY p.product_category_name_english
),
categorias_relevantes AS (
    SELECT
        categoria,
        receita_total
    FROM receita_por_categoria
    WHERE receita_total > 500000
)
SELECT *
FROM categorias_relevantes
ORDER BY receita_total DESC;
"""

banco_dados.sql(query).df()

---

# Slide 8 — Exercícios: CTEs na prática

Reescreva consultas usando `WITH` para deixar a lógica mais clara.

## Slide 8 — Exercício 1 — Vendas por estado

Crie uma CTE chamada `vendas_por_estado` com a quantidade de pedidos e a receita total por estado.

In [55]:
# Resposta do aluno

query = """
WITH  vendas_por_estado AS (
SELECT
    c.customer_state,
    COUNT(f.order_id) as qtd_pedidos,
    SUM(f.valor_total) as receita_total
FROM fato_vendas f
JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY c.customer_state)
SELECT 
* 
FROM vendas_por_estado
ORDER BY receita_total DESC
"""


banco_dados.sql(query).df()

,customer_state,qtd_pedidos,receita_total
0,SP,46448,5769703.15
1,RJ,14143,2055401.57
2,MG,12916,1818891.67
3,RS,6134,861472.79
4,PR,5649,781708.80
5,SC,4097,595127.78
6,BA,3683,591137.81
7,DF,2355,346123.35
8,GO,2277,334212.35
9,ES,2225,317657.93


## Slide 8 — Exercício 2 — Análise por categoria
Crie uma CTE chamada resumo_categorias para calcular a quantidade de vendas, a receita total e o ticket médio por categoria de produto. Depois, na consulta final, mostre apenas as categorias com receita total acima de R$ 100.000.

In [ ]:
# Resposta do aluno

query = """
WITH resumo_categorias AS (
SELECT
    p.product_category_name as categoria,
    COUNT(f.order_id) as qtd_itens,
    SUM(f.valor_total) as receita_total,
    ROUND(AVG(f.valor_total),2) as ticket_medio
FROM fato_vendas f
JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
GROUP BY p.product_category_name
ORDER BY receita_total DESC
)
SELECT *
FROM resumo_categorias
WHERE receita_total > 100_000
"""

banco_dados.sql(query).df()

,categoria,qtd_itens,receita_total,ticket_medio
0,beleza_saude,9465,1412089.53,149.19
1,relogios_presentes,5859,1264333.12,215.79
2,cama_mesa_banho,10953,1225209.26,111.86
3,esporte_lazer,8431,1118256.91,132.64
4,informatica_acessorios,7644,1032723.77,135.10
5,moveis_decoracao,8160,880329.92,107.88
6,utilidades_domesticas,6795,758392.25,111.61
7,cool_stuff,3718,691680.89,186.04
8,automotivo,4140,669454.75,161.70
9,ferramentas_jardim,4268,567145.68,132.88


## Slide 8 — Exercício 3 — Top categorias por estado

Crie CTEs para descobrir quais são as categorias de produto com maior receita em cada estado.
Primeiro, calcule a receita total por estado e categoria. Depois, filtre apenas combinações com receita acima de R$ 50.000 e ordene o resultado por estado e receita.

In [51]:
# Resposta do aluno

query = """
WITH receita_total_estado_categoria AS(
SELECT
    c.customer_state as estado, 
    p.product_category_name as categoria,
    SUM(f.valor_total) as receita
FROM fato_vendas f
JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
GROUP BY c.customer_state, p.product_category_name
)
SELECT * FROM receita_total_estado_categoria
WHERE receita > 50_000
ORDER BY estado, receita
"""

banco_dados.sql(query).df()

,estado,categoria,receita
0,BA,relogios_presentes,50254.70
1,BA,beleza_saude,58620.07
2,MG,perfumaria,53810.26
3,MG,bebes,56088.44
4,MG,brinquedos,66616.00
...,...,...,...
61,SP,informatica_acessorios,386280.65
62,SP,esporte_lazer,428212.45
63,SP,relogios_presentes,448582.76
64,SP,beleza_saude,510255.38


---

# Slide 11 — Conceito: Window Functions

O slide compara `GROUP BY` e `OVER()`.

- `GROUP BY` resume os dados e reduz a quantidade de linhas.
- `OVER()` calcula uma métrica sem apagar o detalhe das linhas.

A partir daqui, os exemplos usam Window Functions.

---

# Slide 12 — Exemplo: Anatomia de uma Window Function

O slide apresenta `PARTITION BY`, `ORDER BY` e a função `ROW_NUMBER()`.

In [ ]:
# Exemplo do Slide 12 adaptado

query = """
SELECT
    f.venda_sk,
    f.order_id,
    p.product_category_name AS categoria,
    f.valor_total,
    ROW_NUMBER() OVER (
        PARTITION BY p.product_category_name
        ORDER BY f.valor_total DESC
    ) AS ranking_venda_categoria
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
WHERE p.product_category_name IS NOT NULL
ORDER BY
    p.product_category_name,
    ranking_venda_categoria;
"""

banco_dados.sql(query).df()

---

# Slide 13 — Exemplo: ROW_NUMBER()

`ROW_NUMBER()` cria uma numeração sequencial para cada linha dentro da partição.

In [ ]:
# Exemplo do Slide 13 adaptado

query = """
WITH ranking_vendas_categoria AS (
    SELECT
        f.venda_sk,
        f.order_id,
        p.product_category_name AS categoria,
        f.valor_total,
        ROW_NUMBER() OVER (
            PARTITION BY p.product_category_name
            ORDER BY f.valor_total DESC
        ) AS posicao
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
)

SELECT
    venda_sk,
    order_id,
    categoria,
    valor_total
FROM ranking_vendas_categoria
WHERE posicao = 1
ORDER BY valor_total DESC;
"""

banco_dados.sql(query).df()

---

# Slide 14 — Exemplo: RANK() e DENSE_RANK()

O slide mostra a diferença entre rankings com empate.

In [ ]:
# Exemplo do Slide 14 adaptado

query = """
WITH receita_por_categoria AS (
    SELECT
        p.product_category_name AS categoria,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY
        p.product_category_name
)

SELECT
    categoria,
    ROUND(receita_total, 2) AS receita_total,

    RANK() OVER (
        ORDER BY receita_total DESC
    ) AS rank_categoria,

    DENSE_RANK() OVER (
        ORDER BY receita_total DESC
    ) AS dense_rank_categoria

FROM receita_por_categoria
ORDER BY receita_total DESC;
"""

banco_dados.sql(query).df()

---

# Slide 15 — Exemplo: SUM() OVER()

O slide apresenta acumulados, médias móveis e percentual do total.

## Slide 15 — Exemplo 1: receita acumulada por estado

In [ ]:
# Exemplo do Slide 15

query = """
SELECT
    p.product_category_name AS categoria,
    f.order_id,
    f.valor_total,

    SUM(f.valor_total) OVER (
        PARTITION BY p.product_category_name
    ) AS total_categoria,

    100.0 * f.valor_total / SUM(f.valor_total) OVER (
        PARTITION BY p.product_category_name
    ) AS percentual_na_categoria

FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
WHERE p.product_category_name IS NOT NULL;

"""

banco_dados.sql(query).df()

---

# Slide 16 — Boas práticas: Otimizando consultas com Window Functions

Antes de usar Window Function em uma tabela muito grande:

1. Filtre os dados antes com `WHERE` ou CTE.
2. Calcule apenas as colunas necessárias.
3. Evite `PARTITION BY` sem necessidade.
4. Observe colunas usadas em `PARTITION BY` e `ORDER BY`.

---

# Slide 17 — Exercícios

Aplique Window Functions para resolver os problemas abaixo.

## Slide 17 — Exercício 1 — Pedido mais caro de cada estado

Use `ROW_NUMBER()` para encontrar o pedido de maior valor em cada estado.

In [ ]:
# Resposta do aluno
query = """

-- coloque sua resposta aqui

"""

banco_dados.sql(query).df()


## Slide 17 — Exercício 2 — Ranking de estados por total vendido

Calcule a receita total por estado e use 'RANK()' para ranquear os estados com maior faturamento.

In [57]:
# Resposta do aluno
query = """

SELECT
    p.product_category_name as categoria,
    SUM(f.valor_total) as receita_total,
RANK() OVER(
ORDER BY receita_total DESC
) as rank
FROM fato_vendas f
JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
GROUP BY p.product_category_name

"""

banco_dados.sql(query).df()


,categoria,receita_total,rank
0,beleza_saude,1412089.53,1
1,relogios_presentes,1264333.12,2
2,cama_mesa_banho,1225209.26,3
3,esporte_lazer,1118256.91,4
4,informatica_acessorios,1032723.77,5
...,...,...,...
69,pc_gamer,1430.10,70
70,casa_conforto_2,1170.58,71
71,cds_dvds_musicais,954.99,72
72,fashion_roupa_infanto_juvenil,598.67,73


## Slide 17 — Exercício 3 — Receita acumulada mensal por estado

Calcule a receita mensal por estado e depois o acumulado mês a mês. Use SUM() OVER()

In [ ]:
#Resposta do aluno

query = """
-- coloque sua resposta aqui
"""

banco_dados.sql(query).df()


### GABARITO EXERCÍCIOS SLIDE 17

In [ ]:
# Gabarito do exercício 1 slide 17

query = """
-- coloque sua resposta aqui

WITH ranking_pedidos_estado AS (
    SELECT
        c.customer_state AS estado,
        f.order_id,
        f.valor_total,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_state
            ORDER BY f.valor_total DESC
        ) AS posicao
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
)

SELECT
    estado,
    order_id,
    ROUND(valor_total, 2) AS valor_total
FROM ranking_pedidos_estado
WHERE posicao = 1
ORDER BY valor_total DESC;


"""

banco_dados.sql(query).df()

In [ ]:
# Gabarito do exercício 2 slide 17

query = """
WITH receita_por_estado AS (
    SELECT
        c.customer_state AS estado,
        SUM(f.valor_total) AS receita_total,
        COUNT(DISTINCT f.order_id) AS qtd_pedidos
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY
        c.customer_state
)

SELECT
    estado,
    qtd_pedidos,
    ROUND(receita_total, 2) AS receita_total,
    RANK() OVER (
        ORDER BY receita_total DESC
    ) AS ranking_estado
FROM receita_por_estado
ORDER BY ranking_estado;
"""

banco_dados.sql(query).df()

In [ ]:
# Gabarito do exercício 3 slide 17

query = """
WITH receita_mensal_estado AS (
    SELECT
        c.customer_state AS estado,
        t.ano,
        t.mes,
        t.ano_mes,
        SUM(f.valor_total) AS receita_mes
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY
        c.customer_state,
        t.ano,
        t.mes,
        t.ano_mes
)

SELECT
    estado,
    ano,
    mes,
    ano_mes,
    receita_mes,
    SUM(receita_mes) OVER (
            PARTITION BY estado
            ORDER BY ano, mes
        ) AS receita_acumulada_estado
FROM receita_mensal_estado
ORDER BY
    estado,
    ano,
    mes;
"""

banco_dados.sql(query).df()

---

# Slide 20 — Exemplo: Índices e Otimização

O slide mostra uma consulta típica que pode se beneficiar de índice.

A consulta original foi corrigida para usar `INNER JOIN` com `dim_cliente` e `dim_tempo`.

In [ ]:
# Exemplo do Slide 20

query = """
SELECT
    f.venda_sk,
    f.order_id,
    f.cliente_sk,
    f.produto_sk,
    f.tempo_sk,
    f.valor_total,
    c.customer_state AS estado,
    t.data AS data_venda
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
WHERE c.customer_state = 'SP'
  AND t.data >= '2018-01-01'
ORDER BY
    t.data,
    f.valor_total DESC;
"""

banco_dados.sql(query).df()

---

# Slide 22 — Exemplo: EXPLAIN antes e depois do índice

Use `EXPLAIN` para observar se o banco faz leitura sequencial ou usa algum índice.

In [ ]:
# Exemplo do Slide 22 adaptado

query = """
EXPLAIN
SELECT *
FROM fato_vendas
WHERE cliente_sk = 5000;
"""

resultado = banco_dados.sql(query).df()

print(resultado["explain_value"][0])

In [ ]:
banco_dados.sql("""
CREATE TABLE fato_vendas_base AS
SELECT *
FROM fato_vendas;
""")

In [ ]:
banco_dados.sql("""
CREATE INDEX idx_fato_vendas_cliente_sk
ON fato_vendas_base (cliente_sk);
""")

In [ ]:


query = """
EXPLAIN
SELECT
    venda_sk,
    order_id,
    cliente_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000;
"""

resultado = banco_dados.sql(query).df()

print(resultado["explain_value"][0])
banco_dados.sql(query).df()

### Tranformando os dataframes em tabelas físicas dentro do DuckDB (antes estavamos usando o register que faz o DuckDB como uma fonte externa, porém a título de didática para indices precisamos fazer essa transformação)

In [ ]:
# Transformando os DataFrames registrados em tabelas físicas no DuckDB
banco_dados.sql("DROP TABLE IF EXISTS fato_vendas_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_cliente_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_produto_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_tempo_base")


banco_dados.sql("""
CREATE TABLE fato_vendas_base AS
SELECT *
FROM fato_vendas
""")

banco_dados.sql("""
CREATE TABLE dim_cliente_base AS
SELECT *
FROM dim_cliente
""")

banco_dados.sql("""
CREATE TABLE dim_produto_base AS
SELECT *
FROM dim_produto
""")

banco_dados.sql("""
CREATE TABLE dim_tempo_base AS
SELECT *
FROM dim_tempo
""")

---

# Slide 24 — Exemplos: Criando índices

O slide apresenta índices simples e compostos.  
Abaixo estão exemplos adaptados para as tabelas do DW.

In [ ]:


banco_dados.sql("""
-- Índice simples em chave de tempo
-- Ajuda em consultas que filtram vendas por período
CREATE INDEX idx_dim_tempo_data
ON dim_tempo_base (data);
""")

banco_dados.sql("""
-- Índice em coluna categórica da dimensão cliente
-- Ajuda em consultas que filtram clientes por estado
CREATE INDEX idx_dim_cliente_estado
ON dim_cliente_base (customer_state);
""")

banco_dados.sql("""
-- Índice composto
-- Ajuda em consultas que filtram por cliente e produto juntos
CREATE INDEX idx_fato_vendas_cliente_produto
ON fato_vendas_base (cliente_sk, produto_sk);
""")

In [ ]:
query ="""
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    t.data
FROM fato_vendas_base f
INNER JOIN dim_tempo_base t
    ON f.tempo_sk = t.tempo_sk
WHERE t.data >= '2018-01-01';
"""

banco_dados.sql(query).df()

In [ ]:

query = """
SELECT
    cliente_sk,
    customer_id,
    customer_state
FROM dim_cliente_base
WHERE customer_state = 'SP';"""

banco_dados.sql(query).df()

In [ ]:

query = """

SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420;

  """

banco_dados.sql(query).df()

---

# Slide 25 — Exercícios: Índices

Identifique a coluna usada no filtro e crie o índice adequado.

Considere a transformação necessário no banco usada antes do inicio da aula aqui no VSCODE.  
Rode as cédulas abaixo

In [ ]:
# Transformando os DataFrames registrados em tabelas físicas no DuckDB

banco_dados.sql("DROP TABLE IF EXISTS fato_vendas_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_cliente_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_produto_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_tempo_base")

banco_dados.sql("""
CREATE TABLE fato_vendas_base AS
SELECT *
FROM fato_vendas
""")

banco_dados.sql("""
CREATE TABLE dim_cliente_base AS
SELECT *
FROM dim_cliente
""")

banco_dados.sql("""
CREATE TABLE dim_produto_base AS
SELECT *
FROM dim_produto
""")

banco_dados.sql("""
CREATE TABLE dim_tempo_base AS
SELECT *
FROM dim_tempo
""")

## Slide 25 — Exercício 1 — Índice para filtro por estado

Consulta base:

```sql
SELECT
    cliente_sk,
    customer_id,
    customer_state
FROM dim_cliente_base
WHERE customer_state = 'SP'

```

Crie um índice adequado.

In [ ]:
# Resposta do aluno

query = """
CREATE INDEX -- resposta aqui

ON dim_cliente_base (resposta_aqui);
"""

## Slide 26 — Exercício 2 — Índice para consulta com JOIN e filtro por data

Como `customer_state` está na dimensão cliente, pense nos índices necessários para uma consulta com `JOIN`.

Consulta base:

```sql
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    t.data
FROM fato_vendas_base f
INNER JOIN dim_tempo_base t
    ON f.tempo_sk = t.tempo_sk
WHERE t.data >= '2018-01-01';

```

In [ ]:
# Resposta do aluno

query = """
CREATE INDEX -- resposta aqui
ON dim_tempo_base (-- resposta aqui);
"""


## Slide 27 — Exercício 3 — Índice composto para filtro combinado

Como `customer_state` está na dimensão cliente, pense nos índices necessários para uma consulta com `JOIN`.

Consulta base:

```sql
SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420;

```

In [ ]:
# Resposta do aluno

query = """
CREATE INDEX --resposta aqui
ON fato_vendas_base (-- resposta_aqui, resposta_aqui)
"""


In [ ]:
query = """SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420
  """

banco_dados.sql(query).df()
